In [2]:
import pandas as pd

In [4]:
df = pd.read_csv("data/conexao-labels-full-4.1nano-p2.csv", sep='|')
print(df.shape)
df.head()

(12287, 3)


,sentence,label,justification
0,E certamente com o risco de ser injusto com al...,Loaded_Language,The sentence uses emotionally charged words li...
1,"Queda de 27,5% em relação ao mesmo período do ...",No_Label,The sentences are factual reports and do not c...
2,"A seguir, são sugeridas algumas medidas de com...",No_Label,The sentence is informational and describes me...
3,Cito onze aqui:\n– PL 6944/2017: impede a limi...,No_Label,"The sentence is informational, listing legisla..."
4,"Um ex-presidente, dos mais populares e populis...",Loaded_Language,The sentence uses emotionally charged words li...


In [13]:
import json
pd.set_option('display.max_colwidth', 200)
# one example for each label
for label in df['label'].unique():
    example = df[df['label'] == label].iloc[0]
    print(f"Label: {label}")
    print(f"Sentence: {example['sentence'][:50]}...")
    print(f"Justification: {example['justification'][:50]}...")
    print("-" * 50)

Label: Loaded_Language
Sentence: E certamente com o risco de ser injusto com alguma...
Justification: The sentence uses emotionally charged words like '...
--------------------------------------------------
Label: No_Label
Sentence: Queda de 27,5% em relação ao mesmo período do ano ...
Justification: The sentences are factual reports and do not conta...
--------------------------------------------------
Label: Flag_Waving
Sentence: Ora, se de modo honesto e digno obtenho meios para...
Justification: The sentence emphasizes the importance of individu...
--------------------------------------------------
Label: Exaggeration-Minimisation
Sentence: Quando reflito acerca dos muitos esforços que real...
Justification: The sentence emphasizes the financial burden and r...
--------------------------------------------------
Label: Repetition
Sentence: INCLUSÃO: Anunciou a inclusão de autistas no Censo...
Justification: The sentence repeats the idea of government action...
----------------------

In [3]:
import pandas as pd
import numpy as np
SEED = 42
np.random.seed(SEED)

local_dataset_path = "data/conexao-labels-full-4.1nano-p2.csv"
online_dataset_path = "https://media.githubusercontent.com/media/kamilyassis/pln/refs/heads/main/2026-03-29_label_datasets/data/out/conexao-labels-full-4.1nano-p2.csv"
in_colab = True

DATASET_ARGS = {
    "filepath_or_buffer": local_dataset_path if not in_colab else online_dataset_path,
    "sep": '|'
}

from sklearn.model_selection import train_test_split

In [ ]:
# ── 7a. Load your data ──────────────────────
df = pd.read_csv(**DATASET_ARGS)
texts  = df['sentence'].tolist()
labels = df['label'].tolist()

id2label = {
    0: "No_Label",
    1: "Loaded_Language",
    2: "Name_Calling-Labeling",
    3: "Doubt",
    4: "Repetition",
    5: "Appeal_to_Fear-Prejudice",
    6: "Flag_Waving",
    7: "Exaggeration-Minimisation"
}
label2id = {v: k for k, v in id2label.items()}

labels = [label2id[l] for l in labels]

# ──────────────────────────────────────────────
# CHECK CLASS IMBALANCE BEFORE
# ──────────────────────────────────────────────
print("=" * 60)
print("CLASS DISTRIBUTION BEFORE OVERSAMPLING")
print("=" * 60)
unique, counts = np.unique(labels, return_counts=True)
for label_id, count in zip(unique, counts):
    print(f"  {id2label[label_id]:30s} → {count:5d} samples")
print(f"  Total: {len(labels)} samples\n")
True
# ──────────────────────────────────────────────
# APPLY RANDOMOVERSAMPLER
# ──────────────────────────────────────────────
from imblearn.over_sampling import RandomOverSampler

# Create 2D array for texts (required by imbalanced-learn)
texts_array = np.array(texts).reshape(-1, 1)

# Oversample to balance all classes
ros = RandomOverSampler(random_state=SEED)
texts_resampled, labels_resampled = ros.fit_resample(texts_array, labels)

# Garantir conversão segura
texts = [str(t).strip() for t in texts_resampled.squeeze()] if isinstance(texts_resampled, np.ndarray) else texts_resampled
labels = list(labels_resampled) if not isinstance(labels_resampled, list) else labels_resampled

# ──────────────────────────────────────────────
# CHECK CLASS IMBALANCE AFTER
# ──────────────────────────────────────────────
print("=" * 60)
print("CLASS DISTRIBUTION AFTER OVERSAMPLING")
print("=" * 60)
unique, counts = np.unique(labels, return_counts=True)
for label_id, count in zip(unique, counts):
    print(f"  {id2label[label_id]:30s} → {count:5d} samples")
print(f"  Total: {len(labels)} samples\n")

# ── 7b. Train / val / test split ────────────
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=SEED, stratify=labels
)

print("=" * 60)
print("TRAIN SET CLASS DISTRIBUTION")
print("=" * 60)
unique, counts = np.unique(train_labels, return_counts=True)
for label_id, count in zip(unique, counts):
    pct = 100 * count / len(train_labels)
    print(f"  {id2label[label_id]:30s} → {count:5d} samples ({pct:5.1f}%)")
print(f"  Total: {len(train_labels)} samples\n")

CLASS DISTRIBUTION BEFORE OVERSAMPLING
  No_Label                       →  5060 samples
  Loaded_Language                →  4252 samples
  Name_Calling-Labeling          →   366 samples
  Doubt                          →   758 samples
  Repetition                     →   234 samples
  Appeal_to_Fear-Prejudice       →   160 samples
  Flag_Waving                    →   696 samples
  Exaggeration-Minimisation      →   761 samples
  Total: 12287 samples

CLASS DISTRIBUTION AFTER OVERSAMPLING
  No_Label                       →  5060 samples
  Loaded_Language                →  5060 samples
  Name_Calling-Labeling          →  5060 samples
  Doubt                          →  5060 samples
  Repetition                     →  5060 samples
  Appeal_to_Fear-Prejudice       →  5060 samples
  Flag_Waving                    →  5060 samples
  Exaggeration-Minimisation      →  5060 samples
  Total: 40480 samples

TRAIN SET CLASS DISTRIBUTION
  No_Label                       →  4048 samples ( 12.5%)
  Lo

In [7]:
train_texts[:3], train_labels[:3]

(['Esta elite islâmica do Norte, é a principal responsável pela introdução da sharia nestas doze províncias.',
  "Quantas vezes pessoas já nos disseram que se tivéssemos 'falado direito' tudo teria sido diferente.",
  'Sim, Dr. Merha, não é mais eficaz, pois são doentes graves, já no estágio 3 hiperinflamatório da doença.'],
 [2, 4, 7])

In [ ]:
# save to csv
train_df = pd.DataFrame({"sentence": train_texts, "label": train_labels})
train_df.to_csv("data/out/train_oversampled.csv", index=False, sep='|')

## tratar g1 faketrue

In [15]:
import pandas as pd

df = pd.read_csv("data/g1_faketrue.csv")
print(df.shape)
df.head()

(1786, 3)


,texto,rotulo,argumento
0,carnaval em olinda. arrastão monstro. fazuele ...,"Xingamento-Rotulação,Apelo_ao_Medo-Preconceito...",O texto utiliza a rotulação ao chamar Lula de ...
1,carro alegórico da escola de samba grande rio...,"Apelo_ao_Medo-Preconceito,Linguagem_Carregada,...",O texto utiliza o medo ao mencionar que há 'ca...
2,cantor léo apoiando atos antidemocráticos. alg...,"Xingamento-Rotulação,Dúvida,Apelo_ao_Medo-Prec...",O texto menciona 'cantor léo apoiando atos ant...
3,versão 1 ore muito pela cantora anitta! ela es...,"Apelo_ao_Medo-Preconceito,Linguagem_Carregada,...",O texto utiliza apelo ao medo ao descrever a g...
4,globo já gravou o vídeo de fim de ano com lula...,"Linguagem_Carregada,Apelo_ao_Medo-Preconceito,...","O texto utiliza linguagens carregadas, como ""v..."


In [16]:
# explode "rotulo" column into multiple rows by splitting on comma
df_exploded = df.assign(rotulo=df['rotulo'].str.split(',')).explode('rotulo').reset_index(drop=True)
print(df_exploded.shape)
df_exploded.head()

(6911, 3)


,texto,rotulo,argumento
0,carnaval em olinda. arrastão monstro. fazuele ...,Xingamento-Rotulação,O texto utiliza a rotulação ao chamar Lula de ...
1,carnaval em olinda. arrastão monstro. fazuele ...,Apelo_ao_Medo-Preconceito,O texto utiliza a rotulação ao chamar Lula de ...
2,carnaval em olinda. arrastão monstro. fazuele ...,Exagero-Minimização,O texto utiliza a rotulação ao chamar Lula de ...
3,carnaval em olinda. arrastão monstro. fazuele ...,Linguagem_Carregada,O texto utiliza a rotulação ao chamar Lula de ...
4,carnaval em olinda. arrastão monstro. fazuele ...,Culpa_por_Associação,O texto utiliza a rotulação ao chamar Lula de ...


In [17]:
df_exploded['rotulo'].value_counts()

rotulo
Apelo_ao_Medo-Preconceito           1232
Linguagem_Carregada                 1049
Repetição                            785
Xingamento-Rotulação                 718
Dúvida                               615
Exagero-Minimização                  581
Culpa_por_Associação                 378
Apelo_à_Popularidade                 306
Apelo_à_Autoridade                   199
Apelo_à_Bandeira                     174
Simplificação_Causal                 156
Ofuscação-Vagueza-Confusão           153
Apelo_à_Hipocrisia                   133
Slogans                              111
Encerrador_de_Debate                  87
Arenque_Vermelho                      66
Falso_Dilema-Sem_Escolha              63
Nenhuma                               45
Espantalho                            23
Whataboutismo                         21
Apelo_à_Authoridade                    3
Oportunidade de Compartilhamento       1
Urgência                               1
Lingüagem_Carregada                    1
Linguaem_

In [18]:
labels_to_keep = [
    "Apelo_ao_Medo-Preconceito",
    "Linguagem_Carregada",
    "Repetição",
    "Xingamento-Rotulação",
    "Dúvida",
    "Exagero-Minimização",
    "Apelo_à_Bandeira",
    "Nenhuma"
]

df_filtered = df_exploded[df_exploded['rotulo'].isin(labels_to_keep)].reset_index(drop=True)
print(df_filtered.shape)
df_filtered.head()

(5199, 3)


,texto,rotulo,argumento
0,carnaval em olinda. arrastão monstro. fazuele ...,Xingamento-Rotulação,O texto utiliza a rotulação ao chamar Lula de ...
1,carnaval em olinda. arrastão monstro. fazuele ...,Apelo_ao_Medo-Preconceito,O texto utiliza a rotulação ao chamar Lula de ...
2,carnaval em olinda. arrastão monstro. fazuele ...,Exagero-Minimização,O texto utiliza a rotulação ao chamar Lula de ...
3,carnaval em olinda. arrastão monstro. fazuele ...,Linguagem_Carregada,O texto utiliza a rotulação ao chamar Lula de ...
4,carro alegórico da escola de samba grande rio...,Apelo_ao_Medo-Preconceito,O texto utiliza o medo ao mencionar que há 'ca...


In [19]:
# drop duplicates, select first labels (occurrence of text) only
df_final = df_filtered.drop_duplicates(subset=['texto'], keep='first').reset_index(drop=True)
print(df_final.shape)
df_final.head()

(1747, 3)


,texto,rotulo,argumento
0,carnaval em olinda. arrastão monstro. fazuele ...,Xingamento-Rotulação,O texto utiliza a rotulação ao chamar Lula de ...
1,carro alegórico da escola de samba grande rio...,Apelo_ao_Medo-Preconceito,O texto utiliza o medo ao mencionar que há 'ca...
2,cantor léo apoiando atos antidemocráticos. alg...,Xingamento-Rotulação,O texto menciona 'cantor léo apoiando atos ant...
3,versão 1 ore muito pela cantora anitta! ela es...,Apelo_ao_Medo-Preconceito,O texto utiliza apelo ao medo ao descrever a g...
4,globo já gravou o vídeo de fim de ano com lula...,Linguagem_Carregada,"O texto utiliza linguagens carregadas, como ""v..."


In [20]:
df_final = df_final.rename(columns={"texto": "sentence", "rotulo": "label", "argumento": "justification"})

In [22]:
translation_labelling_map = {
    "Apelo_ao_Medo-Preconceito": "Appeal_to_Fear-Prejudice",
    "Linguagem_Carregada": "Loaded_Language",
    "Repetição": "Repetition",
    "Xingamento-Rotulação": "Name_Calling-Labeling",
    "Dúvida": "Doubt",
    "Exagero-Minimização": "Exaggeration-Minimisation",
    "Apelo_à_Bandeira": "Flag_Waving",
    "Nenhuma": "No_Label"
}

df_final['label'] = df_final['label'].map(translation_labelling_map)
print(df_final['label'].value_counts())

label
Appeal_to_Fear-Prejudice     706
Loaded_Language              402
Name_Calling-Labeling        275
Repetition                   114
Doubt                         85
Exaggeration-Minimisation     68
Flag_Waving                   53
No_Label                      44
Name: count, dtype: int64


In [23]:
df_final.to_csv("data/out/g1_faketrue_treated.csv", index=False, sep='|')